# 📓 Notebook 7 — NumPy Fundamentals

> **Module:** Data Science Libraries · **Estimated time:** 30–40 min · **Difficulty:** Beginner / Intermediate

NumPy ("Numerical Python") is the bedrock of the entire Python data-science ecosystem. **Pandas, scikit-learn, TensorFlow, PyTorch, OpenCV** — all of them store data as NumPy arrays under the hood. Mastering NumPy is the single biggest leverage move in the early stages of a data-science journey.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. Explain *why* NumPy exists (speed + expressiveness vs. plain Python lists).
2. Create arrays from scratch and from existing data.
3. Inspect arrays via `shape`, `dtype`, `ndim`, `size`.
4. **Index and slice** 1-D and 2-D arrays, including boolean masks.
5. Perform **vectorised** arithmetic and use universal functions (`np.sin`, `np.exp`, …).
6. Understand **broadcasting** — the most important NumPy concept after slicing.
7. Compute statistics over rows / columns with `axis`.
8. Generate reproducible **random data** with `np.random.default_rng`.
9. Reshape, stack, and split arrays.

## ✅ Prerequisites

Notebooks 1–6 (lists, comprehensions, functions, imports).

## 1. Why NumPy?

For small datasets pure Python is fine. For *any* serious numerical work, three problems with Python lists become unbearable:

1. **Slow** — Python loops are interpreted; NumPy runs vectorised C code under the hood (often 50–500× faster).
2. **Memory-hungry** — every `int` in a Python list is a full object; a NumPy array of 1 million `int32` uses ~4 MB instead of ~28 MB.
3. **Awkward syntax** — element-wise arithmetic on lists requires comprehensions; NumPy makes it as natural as scalar maths.

Let\'s see all three in action.

In [ ]:
import numpy as np

# Big list vs. big array
py_list = list(range(1_000_000))
np_arr  = np.arange(1_000_000)

import time

# Sum a million numbers, the pure-Python way
t0 = time.perf_counter()
s1 = sum(x*x for x in py_list)
t_py = time.perf_counter() - t0

# Sum a million numbers, NumPy-style (vectorised)
t0 = time.perf_counter()
s2 = (np_arr * np_arr).sum()
t_np = time.perf_counter() - t0

assert s1 == s2
print(f"Pure Python : {t_py*1000:7.1f} ms")
print(f"NumPy       : {t_np*1000:7.1f} ms   ({t_py/t_np:.0f}x faster)")

## 2. Creating arrays

There are a handful of constructors you will use constantly:

| Call                        | Result                                    |
|-----------------------------|-------------------------------------------|
| `np.array([1, 2, 3])`       | from a Python list                        |
| `np.zeros((3, 4))`          | shape `(3, 4)`, all zeros                 |
| `np.ones((2, 5))`           | shape `(2, 5)`, all ones                  |
| `np.full((2, 2), 7)`        | filled with 7                             |
| `np.arange(0, 10, 2)`       | like `range`, but as an array              |
| `np.linspace(0, 1, 5)`      | 5 numbers evenly spaced in [0, 1]         |
| `np.eye(3)`                 | 3×3 identity matrix                       |
| `rng.random((rows, cols))`  | random values in [0, 1)                   |

In [ ]:
import numpy as np

a = np.array([1, 2, 3, 4, 5])
b = np.zeros((3, 4))
c = np.ones((2, 5))
d = np.full((2, 2), 7)
e = np.arange(0, 10, 2)
f = np.linspace(0, 1, 5)
g = np.eye(3)

for name, arr in [("a", a), ("b", b), ("c", c), ("d", d), ("e", e), ("f", f), ("g", g)]:
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}")
    print(arr)
    print()

## 3. Inspecting an array

The four attributes you check first:

- `shape` — a tuple `(rows, cols, ...)`.
- `dtype` — element data type (`int64`, `float64`, `bool`, …).
- `ndim` — number of dimensions.
- `size` — total number of elements.

In [ ]:
A = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]])

print(f"A:\n{A}\n")
print(f"shape : {A.shape}")     # (3, 4)
print(f"dtype : {A.dtype}")
print(f"ndim  : {A.ndim}")
print(f"size  : {A.size}")
print(f"max   : {A.max()}")
print(f"min   : {A.min()}")
print(f"mean  : {A.mean()}")
print(f"sum   : {A.sum()}")

> 💡 **In machine learning, `shape` is everything.** Most cryptic errors ("expected (32, 4) got (4, 32)") are shape mismatches. When you debug numerical code, *always* `print(x.shape)` first.

## 4. Indexing and slicing

For 1-D arrays this is identical to Python lists. For 2-D, the syntax is `A[row, col]` with slicing on each axis.

In [ ]:
A = np.arange(1, 21).reshape(4, 5)
print(A)
print()

# Single element
print(f"A[0, 0]   = {A[0, 0]}")
print(f"A[-1, -1] = {A[-1, -1]}")

# Whole row / column
print(f"\nRow 0    : {A[0]}")
print(f"Row 0    : {A[0, :]}    (explicit form)")
print(f"Col 2    : {A[:, 2]}")
print(f"Last col : {A[:, -1]}")

# Sub-matrix
print(f"\nA[1:3, 0:2]:")
print(A[1:3, 0:2])

### Boolean masking — the most powerful indexing pattern

A boolean array of the same shape can be used to index — only the `True` positions are returned. This is how you do "give me all rows where age > 30" in pure NumPy and (with extra wrapping) in pandas.

In [ ]:
x = np.array([10, 25, 7, 18, 33, 2, 41, 9])
print(f"x = {x}")

mask = x > 15
print(f"x > 15 → mask = {mask}")
print(f"x[mask]      = {x[mask]}")

# In one step — the idiomatic form
print(f"x[x > 15]    = {x[x > 15]}")

# Combine masks with & (AND) and | (OR) — parentheses required
print(f"\n10 ≤ x ≤ 30 : {x[(x >= 10) & (x <= 30)]}")

## 5. Vectorised arithmetic

This is what makes NumPy *feel* like maths: you write the formula once, NumPy applies it to every element. No loops.

In [ ]:
temps_c = np.array([0, 10, 20, 25, 30, 35, 100], dtype=float)
print(f"°C : {temps_c}")

# Convert ALL to Fahrenheit in one expression — no loop required
temps_f = temps_c * 9 / 5 + 32
print(f"°F : {temps_f}")

# Element-wise operations between arrays of the same shape
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])
print(f"\na + b = {a + b}")
print(f"a * b = {a * b}")
print(f"b / a = {b / a}")

### Universal functions (ufuncs)

NumPy provides element-wise versions of every common math function. They are *fast* and they handle entire arrays at once.

In [ ]:
x = np.linspace(0, 2 * np.pi, 5)
print(f"x       : {x}")
print(f"sin(x)  : {np.sin(x)}")
print(f"cos(x)  : {np.cos(x)}")
print(f"exp(x)  : {np.exp(x)}")
print(f"log(1+x): {np.log1p(x)}")
print(f"sqrt(x) : {np.sqrt(x)}")

## 6. Broadcasting — the rule that powers most of NumPy

What if two arrays have *different* shapes? NumPy tries to *stretch* the smaller one to match. The rule is:

> Compare shapes from the **right**. Dimensions are compatible when they are **equal** or one of them is **1**.

Two examples make this concrete.

In [ ]:
# Example 1: array + scalar (scalar broadcast over every element)
a = np.array([1, 2, 3, 4])
print(f"a + 100 = {a + 100}")

# Example 2: row vector + matrix
# A: shape (3, 4); row: shape (4,)
# Broadcasting expands row to (1, 4) → (3, 4) to match A
A   = np.arange(1, 13).reshape(3, 4)
row = np.array([10, 20, 30, 40])
print(f"\nA:\n{A}")
print(f"\nrow: {row}")
print(f"\nA + row:\n{A + row}")    # row added to EACH row

# Example 3: column vector + matrix
col = np.array([[100], [200], [300]])    # shape (3, 1)
print(f"\ncol:\n{col}")
print(f"\nA + col:\n{A + col}")    # col added to EACH column

> 🎯 **Mental model.** "Compatible shapes" means: line them up from the right; each pair of dimensions must be equal, or one must be 1. Then the array with the 1 gets stretched to match.

This single rule replaces what would otherwise be hundreds of nested loops.

## 7. Aggregations along an axis

Sum, mean, max, min — these all work over the whole array by default, but you can ask for them along a specific **axis**.

- `axis=0` → collapse rows → one value per column.
- `axis=1` → collapse columns → one value per row.

```
        col 0  col 1  col 2  col 3
row 0     1     2     3     4
row 1     5     6     7     8     ← sum over axis=1 (each row → 1 number)
row 2     9    10    11    12

         ↓  sum over axis=0 (each col → 1 number)
        15   18   21   24
```

In [ ]:
A = np.arange(1, 13).reshape(3, 4).astype(float)
print(f"A:\n{A}\n")

print(f"sum()          : {A.sum()}              (everything)")
print(f"sum(axis=0)    : {A.sum(axis=0)}   (column sums)")
print(f"sum(axis=1)    : {A.sum(axis=1)}        (row sums)")
print()
print(f"mean(axis=0)   : {A.mean(axis=0)}")
print(f"argmax(axis=1) : {A.argmax(axis=1)}        (index of max in each row)")

## 8. Reshape, stack, split

Manipulating shapes is one of the most common operations in ML.

In [ ]:
x = np.arange(12)
print(f"x : shape={x.shape} → {x}")

# reshape to whatever you need (must match total size)
m34 = x.reshape(3, 4)
m43 = x.reshape(4, 3)
m26 = x.reshape(2, 6)
print(f"\nreshape (3,4):\n{m34}")
print(f"\nreshape (4,3):\n{m43}")

# Use -1 to let NumPy figure out one dimension
auto = x.reshape(-1, 4)
print(f"\nreshape (-1,4):\n{auto}   shape={auto.shape}")

# Flatten back to 1-D
print(f"\nflatten        : {m34.flatten()}")
print(f"ravel          : {m34.ravel()}    (same, but a view when possible)")

In [ ]:
# Stacking arrays
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("hstack (horizontal):", np.hstack([a, b]))
print("vstack (vertical) :\n", np.vstack([a, b]))
print("stack new axis    :\n", np.stack([a, b]))   # shape (2, 3)
print("column_stack      :\n", np.column_stack([a, b]))   # shape (3, 2)

## 9. Random data — reproducibly

For experiments and demos you want random data that *isn\'t actually random*: you want the same numbers every time you run the notebook so results are reproducible. Use `np.random.default_rng(seed)`.

In [ ]:
rng = np.random.default_rng(seed=42)

print("rng.random((3, 4)):")
print(rng.random((3, 4)))

print("\nrng.integers(low=0, high=10, size=(2, 5)):")
print(rng.integers(0, 10, size=(2, 5)))

print("\nrng.normal(mean=50, std=15, size=8):")
print(rng.normal(50, 15, size=8).round(2))

print("\nrng.choice(['cat','dog','bird'], size=5):")
print(rng.choice(["cat", "dog", "bird"], size=5))

> 💡 The old `np.random.seed` / `np.random.rand` API still works but the **Generator API** (`default_rng`) is the recommended modern way: explicit, isolated, and supports parallel streams.

## 10. Putting it all together — a tiny applied example

Suppose we have **5 students** each taking **3 exams**. We want to:

1. compute each student\'s **average** (per-row),
2. each exam\'s **average** (per-column),
3. apply a **+5 bonus** to exam 2 only, then re-compute,
4. find the **top-3 students** by overall average.

This combines indexing, broadcasting, axis aggregation, and `argsort`.

In [ ]:
rng = np.random.default_rng(0)

# 5 students × 3 exams (scores 0-100)
scores = rng.integers(40, 100, size=(5, 3))
students = np.array(["Alice", "Bob", "Carol", "Dan", "Eva"])
exams    = np.array(["Math", "Sci", "Eng"])

print(f"      {exams[0]:>4} {exams[1]:>4} {exams[2]:>4}")
for name, row in zip(students, scores):
    print(f"{name:<6}{row[0]:>5}{row[1]:>5}{row[2]:>5}")

# 1. per-student average
student_avg = scores.mean(axis=1)
print("\nStudent averages:")
for name, avg in zip(students, student_avg):
    print(f"  {name:<6} {avg:5.2f}")

# 2. per-exam average
exam_avg = scores.mean(axis=0)
print("\nExam averages:", dict(zip(exams, exam_avg.round(2))))

# 3. +5 bonus on exam 2 (Eng) only — broadcasting an array of [0, 0, 5]
bonus = np.array([0, 0, 5])
new_scores = np.clip(scores + bonus, 0, 100)
print(f"\nAfter +5 English bonus:\n{new_scores}")

# 4. top-3 by overall average
new_avg = new_scores.mean(axis=1)
order = np.argsort(-new_avg)             # descending
print("\nRanking after bonus:")
for rank, idx in enumerate(order, start=1):
    print(f"  {rank}. {students[idx]:<6}  avg = {new_avg[idx]:5.2f}")

**What just happened?**

- We used **shape** explicitly: `(5, 3)`. Rows are students, columns are exams.
- `axis=1` gave us per-row (per-student) means; `axis=0` gave us per-column (per-exam) means.
- `bonus = np.array([0, 0, 5])` was **broadcast** across all 5 rows.
- `np.clip` capped the result at 100.
- `np.argsort(-new_avg)` returned the indices that would sort the array in descending order — the ranking.

That\'s a tiny taste of how NumPy thinks. You will see exactly these patterns in scikit-learn (Notebook 9) where rows are *samples* and columns are *features*.

## 🧪 Practice exercises

### Exercise 1 — Create and slice

1. Create a 4×4 array containing the numbers 1–16.
2. Print the second row.
3. Print the third column.
4. Print the **bottom-right 2×2** sub-matrix.

In [ ]:
# Your code here  👇
import numpy as np


<details>
<summary>💡 <b>Solution</b></summary>

```python
A = np.arange(1, 17).reshape(4, 4)
print(A)
print()
print("row 1 :", A[1])
print("col 2 :", A[:, 2])
print("bottom-right 2x2:\n", A[-2:, -2:])
```
</details>

### Exercise 2 — Vectorised z-score

Given a 1-D array `x`, compute its **z-score**: subtract the mean, divide by the standard deviation. Verify that the result has mean ≈ 0 and std ≈ 1.

In [ ]:
# Your code here  👇
x = np.array([12.0, 18.0, 17.0, 22.0, 25.0, 14.0, 19.0, 21.0])


<details>
<summary>💡 <b>Solution</b></summary>

```python
z = (x - x.mean()) / x.std()
print("z         :", z.round(3))
print("z.mean()  :", z.mean())
print("z.std()   :", z.std())
```

Standardising features so they have mean 0 and std 1 is a *very* common preprocessing step in ML.
</details>

### Exercise 3 — Boolean filtering

Given an array of daily temperatures, print:

1. The number of *hot* days (temperature > 30).
2. The mean temperature of *cold* days (< 10).
3. A new array where any temperature above 40 is clipped to 40.

In [ ]:
# Your code here  👇
temps = np.array([5, 25, 35, 12, 28, 42, 3, 19, 31, 8, 22, 45])


<details>
<summary>💡 <b>Solution</b></summary>

```python
n_hot = (temps > 30).sum()
print(f"Hot days     : {n_hot}")

cold_mean = temps[temps < 10].mean()
print(f"Mean of cold : {cold_mean:.2f}")

clipped = np.clip(temps, None, 40)
print(f"Clipped      : {clipped}")
```

**Pattern.** `(condition).sum()` counts how many entries satisfy a condition — the boolean array of True/False is summed as 1/0.
</details>

### Exercise 4 — Broadcasting in action

You have a 10×3 matrix `X` of features and want to **standardise each column** to have mean 0 and std 1 — i.e. subtract the column mean and divide by the column std. Do it in **one expression** using broadcasting (no loops).

In [ ]:
# Your code here  👇
rng = np.random.default_rng(0)
X = rng.normal(loc=[5, 100, 0.5], scale=[1, 25, 0.1], size=(10, 3))
print("Original X:\n", X.round(3))
print("\nCol means :", X.mean(axis=0).round(3))
print("Col stds  :", X.std(axis=0).round(3))


<details>
<summary>💡 <b>Solution</b></summary>

```python
X_std = (X - X.mean(axis=0)) / X.std(axis=0)
print("Standardised X (rounded):\n", X_std.round(3))
print("\nCheck means :", X_std.mean(axis=0).round(6))
print("Check stds  :", X_std.std(axis=0).round(6))
```

`X.mean(axis=0)` has shape `(3,)`, and broadcasting stretches it across all 10 rows — same for the std. Mean and std of the result should be (almost) 0 and 1 per column.
</details>

### Exercise 5 — Debug me 🐞

The cell below should compute the **column means** but the output looks suspicious. Find and fix the bug.

In [ ]:
A = np.arange(1, 13).reshape(3, 4)
print("A:\n", A)

means = A.mean(axis=1)         # bug
print("Column means:", means)


<details>
<summary>💡 <b>Solution</b></summary>

`axis=1` collapses **columns** to produce a value per **row** — i.e. row means, not column means. For column means use `axis=0`.

```python
means = A.mean(axis=0)
print("Column means:", means)
```

The classic mnemonic: `axis=k` means *"this axis disappears"*. Row index is axis 0, column index is axis 1.
</details>

## 🎁 Bonus mini-project — Simulating a fair coin

Use `rng.integers(0, 2, size=n)` to simulate n coin flips (0 = tails, 1 = heads). For `n = 10`, `n = 1_000`, `n = 100_000`:

1. Print the empirical proportion of heads.
2. Show how this converges to 0.5 as n grows (the **law of large numbers**).
3. Bonus: plot the **running proportion** vs flip number.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

for n in [10, 1_000, 100_000]:
    flips = rng.integers(0, 2, size=n)
    print(f"n={n:>6}: P(heads) ≈ {flips.mean():.4f}")

# Running proportion plot
n = 5_000
flips = rng.integers(0, 2, size=n)
running = np.cumsum(flips) / np.arange(1, n + 1)

plt.figure(figsize=(8, 4))
plt.axhline(0.5, color="red", ls="--", label="True P=0.5")
plt.plot(running, lw=1.2)
plt.title("Law of large numbers: running proportion of heads")
plt.xlabel("Number of flips")
plt.ylabel("Proportion of heads")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
```
</details>

## 🧠 Key takeaways

1. NumPy arrays are **homogeneous**, **typed**, and **vectorised** — that\'s why they\'re fast.
2. Always know your array\'s `shape` and `dtype`; most ML bugs are shape mismatches.
3. **Indexing / slicing** works on each axis; boolean masks let you filter.
4. **Vectorised arithmetic** + **ufuncs** replace explicit loops — write the formula, not the loop.
5. **Broadcasting** stretches arrays of compatible shapes — no for-loops needed for row/column operations.
6. Aggregate along an axis: `axis=0` collapses rows (one value per column), `axis=1` collapses columns.
7. Use **`np.random.default_rng(seed)`** for reproducible random data.

## ✅ Self-assessment

- [ ] Build a 2-D array and inspect `shape`, `dtype`, `ndim`, `size`.
- [ ] Slice a sub-matrix and select a column.
- [ ] Filter with a boolean mask.
- [ ] Compute z-scores via broadcasting.
- [ ] Use `axis=0` vs `axis=1` correctly for aggregations.
- [ ] Reshape and stack arrays.

## 🚀 Next step

Continue with **Notebook 8 — Matplotlib Basics** to turn these arrays into beautiful, informative plots.